# Lab 1 — Probability, Odds & Log-Odds (Logit)

**Day 03 · Classification & Model Interpretation · Cisco AI/ML Training**

---

## Learning objectives

1. Compute **probability** from observed default rates in Lending Club data.
2. Convert probability to **odds** and **log-odds (logit)**.
3. Apply the **inverse logit** to recover probability.
4. Connect these quantities to **logistic regression** (Lab 2).

> **Checkpoints:** **1,000** rows · P(default) ≈ **0.49** · odds ≈ **0.94** · logit ≈ **-0.06**

**Dataset:** Lending Club sample (**1,000** rows)  
**Companion script:** `../scripts/lab01_probability_exercises.py`


## From regression to classification

| Day 2 (Zomato) | Day 3 (Lending Club) |
|----------------|----------------------|
| Target: continuous `aggregate_rating` | Target: binary `default` |
| Model: Linear Regression | Model: Logistic Regression (Lab 2) |
| Metric: RMSE, R² | Metric: precision, recall, AUC (Labs 3–4) |

Today we build the **probability language** logistic regression uses internally.


### Key definitions

| Term | Symbol | Formula | Range |
|------|--------|---------|-------|
| Probability | $p$ | successes / total | 0 to 1 |
| Odds | — | $p / (1-p)$ | 0 to ∞ |
| Log-odds (logit) | — | $\ln(p / (1-p))$ | −∞ to +∞ |

**Intuition:** Odds = 1 means 50/50. Odds = 2 means 2:1 in favor. Logistic regression models log-odds as a **linear** function of features.


---

## 1. Load Lending Club data and define `default`


In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-03":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "lending-club" / "lending_club_sample.csv").is_file():
            GH_ROOT = parent
            break

LENDING_CLUB_CSV = GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv"
DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}

df = pd.read_csv(LENDING_CLUB_CSV)
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

print(f"rows: {len(df)}")
print(df["loan_status"].value_counts())


We label **`default = 1`** for distressed loans (`Charged Off` or `Late`). All other statuses are **0** (paid or current).

This is a simplification of the full Kaggle Lending Club schema — appropriate for classroom classification.


---

## 2. Empirical probability of default


In [ ]:
p_default = df["default"].mean()
p_paid = 1 - p_default

print(f"P(default): {p_default:.4f}")
print(f"P(fully current/paid): {p_paid:.4f}")

assert len(df) == 1000
print("✓ Row count checkpoint OK")


In [ ]:
# Visualize class balance
fig, ax = plt.subplots(figsize=(5, 3))
sns.countplot(data=df, x="default", ax=ax, palette=["steelblue", "salmon"])
ax.set_xticks([0, 1])
ax.set_xticklabels(["No default (0)", "Default (1)"])
ax.set_title("Class balance — nearly 50/50 in lab sample")
plt.tight_layout()
plt.show()


Unlike Day 6 credit-card fraud (**99:1** imbalance), our Lending Club lab sample is **roughly balanced** — accuracy is more meaningful here.


---

## 3. Odds of default

$$
	ext{odds} = rac{p}{1-p}
$$


In [ ]:
odds_default = p_default / p_paid
print(f"odds of default: {odds_default:.4f}")

# Manual check
manual_odds = df["default"].sum() / (len(df) - df["default"].sum())
print(f"manual odds (defaults / non-defaults): {manual_odds:.4f}")
assert abs(odds_default - manual_odds) < 0.001


**Read the number:** odds ≈ **0.94** means slightly **fewer** defaults than non-defaults (odds < 1).


---

## 4. Log-odds (logit)


In [ ]:
log_odds = np.log(odds_default)
print(f"log-odds (logit): {log_odds:.4f}")


Negative logit → odds < 1 → probability < 0.5. Positive logit → probability > 0.5.


---

## 5. Inverse logit — recover probability

Logistic regression outputs log-odds $\eta$; we convert back with the **sigmoid**:

$$
p = rac{1}{1 + e^{-\eta}}
$$


In [ ]:
p_recovered = 1 / (1 + np.exp(-log_odds))
print(f"probability recovered from logit: {p_recovered:.4f}")
print(f"original P(default):           {p_default:.4f}")
assert abs(p_recovered - p_default) < 1e-9
print("✓ Inverse logit matches")


---

## 6. Worked example — P = 0.20


In [ ]:
p_example = 0.20
odds_example = p_example / (1 - p_example)
logit_example = np.log(odds_example)

print(f"example P=0.20 -> odds: {odds_example:.4f}")
print(f"logit: {logit_example:.4f}")

# Recover
p_back = 1 / (1 + np.exp(-logit_example))
print(f"recovered p: {p_back:.2f}")


At **20%** default probability, odds are **0.25** (1:4) and logit is negative — a lender would treat this as lower risk than our portfolio average (~49%).


---

## 7. Bridge to Lab 2

Logistic regression assumes:

$$
	ext{logit}(p) = eta_0 + eta_1 x_1 + eta_2 x_2 + \cdots
$$

Lab 1 used **one** overall rate $p$ for the whole portfolio. Lab 2 estimates **different** $p$ per loan based on `int_rate`, `dti`, etc.


---

## 8. Final checkpoint


In [ ]:
print("Lab 1 — Probability exercises")
print(f"rows: {len(df)}")
print(f"P(default): {p_default:.4f}")
print(f"odds of default: {odds_default:.4f}")
print(f"log-odds (logit): {log_odds:.4f}")
print(f"example P=0.20 -> odds: {odds_example:.4f}")

assert len(df) == 1000
assert 0.45 < p_default < 0.52
print("\n✓ All checkpoint assertions passed")


---

## Reflection questions

1. If P(default) rises from 0.49 to 0.60, do odds increase or decrease?
2. Why does logistic regression model **log-odds** instead of probability directly?
3. Which loan statuses did we map to `default = 1` and why?

**Next:** [Lab 2 — Logistic regression](lab02_logistic_regression.ipynb)
